2. Naïve Bayes — NumPy only

    Binarize each pixel (1 if pixel > 0.5, else 0).
    Estimate conditional probabilities of each pixel being “on” given a digit class.
    Apply Bayes’ rule with the independence assumption.

Sources
https://www.youtube.com/watch?v=Yu_eqv8gATs

In [37]:
import pandas as pd
import seaborn as sns
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, classification_report

In [38]:
loaded = np.load("x_short_data.npz")
x_data = loaded["data"]

loaded = np.load("y_short_target.npz")
y_target = loaded["data"]

In [39]:
# normalize
x_data = x_data / 255
# y_target = y_target.ravel()

In [40]:
x_train, x_test, y_train, y_test = train_test_split(
    x_data, y_target, test_size=0.3, shuffle=True
)

In [41]:
class NaiveBayesOld:
    def fit(self, x_train, y_train):
        self.classes = np.unique(y_train)
        self.priors = [len(y_train[y_train == c]) / len(y_train) for c in self.classes]
        self.means = [x_train[y_train == c].mean() for c in self.classes]
        self.means = [x_train[y_train == c].std() for c in self.classes]
    def compute_likelihood(self, row, class_idx):
        likelihood = 1
        for feature in row.index:
            mean = self.means[class_idx][feature]
            std = self.stds[class_idx][feature]
            likelihood *= (1 / (np.sqrt(2 * np.pi) * std)) * np.exp((-(row[feature] - mean)**2) / (2 * std**2))
        return likelihood
    
    def predict(self, x):
        y_pred = []
        for _, row in x.iterrows():
            posteriors = []
            for i in range(len(self.classes)):
                likelihood = self.compute_likelihood(row, i)
                posteriors.append(likelihood * self.priors[i])
                
            y_pred.append(self.classes[np.argmax(posteriors)])
            
        return np.array(y_pred)
            

In [42]:
class NaiveBayes:
    def fit(self, x_train, y_train):
        x_train = (np.asarray(x_train) > 0.5).astype(np.float64)
        y_train = np.asarray(y_train).ravel()

        self.classes = np.unique(y_train)
        class_counts = np.array([np.sum(y_train == c) for c in self.classes])

        # Laplace smoothing for the probability that each pixel is on.
        self.feature_probs = np.array(
            [
                (x_train[y_train == c].sum(axis=0) + 1) / (class_counts[i] + 2)
                for i, c in enumerate(self.classes)
            ]
        )
        self.log_priors = np.log(class_counts / len(y_train))

    def predict(self, x):
        x = (np.asarray(x) > 0.5).astype(np.float64)

        log_probs = (
            x @ np.log(self.feature_probs).T
            + (1 - x) @ np.log(1 - self.feature_probs).T
            + self.log_priors
        )
        return self.classes[np.argmax(log_probs, axis=1)]

In [43]:
model = NaiveBayes()
model.fit(x_train, y_train)
predictions = model.predict(x_test)

accuracy = np.mean(predictions == y_test) * 100
print(f'Accuracy: {accuracy:.2f}%')

Accuracy: 83.43%
